# Time Series Forecasting for Energy Analytics - NSP Dataset

## Objective
Build Prophet and LSTM forecasting models for all 5 regions using engineered features

## Setup and Imports

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from prophet import Prophet
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
import warnings
# warnings.filterwarnings('ignore')

In [8]:
# path configurations
# Paths
ROOT_DIR = Path.cwd().parent
DATA_DIR = ROOT_DIR / "data"
OUTPUT_DIR = ROOT_DIR / "output"

In [9]:
# Helper functions
model_results = {'Model': [], 'Region': [], 'MAE': [], 'RMSE': [], 'MAPE': []}


# metrics to be stored
def calculate_metrics(y_true, y_pred, model_name, region_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)

    model_results['Model'].append(model_name)
    model_results['Region'].append(region_name)
    model_results['MAE'].append(mae)
    model_results['RMSE'].append(rmse)
    model_results['MAPE'].append(mape)

    print(f"{model_name} - {region_name}: MAE={mae:.2f}, RMSE={rmse:.2f}, MAPE={mape:.2f}%")
    return mae, rmse, mape


# save and show the plot
def save_and_show_plot(fig, filename):
    output_path = OUTPUT_DIR / filename
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"Saved: {output_path}")
    plt.show()


# check for GPU / CPU
def get_device():
    try:
        if torch.cuda.is_available():
            device = torch.device('cuda')
            # Test CUDA by running a small tensor operation
            test_tensor = torch.tensor([1.0, 2.0]).to(device)
            print(f"Test Tensor result: {test_tensor}")
            print(f"CUDA is working. Using device: {device}")
        else:
            device = torch.device('cpu')
            print("CUDA not available, using CPU")
    except Exception as ex:
        device = torch.device('cpu')
        print(f"Error initializing CUDA: {ex}")
        print("Falling back to CPU")
    return device

## Load Engineered Features Dataset

In [21]:
df = pd.read_csv(OUTPUT_DIR / "engineered_features.csv", parse_dates=["timestamp"])
display(df.columns)

display(df.shape)

Index(['timestamp', 'hour', 'day_of_week', 'month', 'year', 'week',
       'is_weekend', 'season', 'is_holiday', 'temperature_c', 'feels_like_c',
       'humidity_pct', 'wind_speed_kmh', 'precipitation_mm', 'cloud_cover_pct',
       'renewable_pct', 'consumption_kwh', 'price_per_kwh', 'grid_load_pct',
       'co2_kg', 'demand_response', 'power_outage', 'anomaly_flag',
       'anomaly_type', 'peak_demand_flag', 'consumption_kwh_lag_1h',
       'consumption_kwh_lag_24h', 'hdd', 'region_Annapolis Valley',
       'region_Cape Breton', 'region_Halifax', 'region_Pictou County',
       'region_South Shore', 'customer_type_Commercial/Industrial',
       'customer_type_Mixed', 'customer_type_Residential'],
      dtype='str')

(438240, 36)

In [22]:
# Verify key features exist
required_features = ['consumption_kwh_lag_1h', 'consumption_kwh_lag_24h',
                     'rolling_mean_168h', 'rolling_std_24h', 'hdd']

print("\nRequired features present:")
for feat in required_features:
    exists = feat in df.columns
    print(f"  {feat}: {'✓' if exists else '✗ MISSING'}")

# Handle region column (check if one-hot encoded or categorical)
if 'region' not in df.columns:
    region_cols = [col for col in df.columns if col.startswith('region_')]
    if region_cols:
        df['region'] = df[region_cols].idxmax(axis=1).str.replace('region_', '')
        print("\nReconstructed 'region' column from one-hot encoding")

print(f"\nRegions: {sorted(df['region'].unique())}")
print("\nSample data:")
df[['timestamp', 'region', 'consumption_kwh', 'consumption_kwh_lag_24h',
    'rolling_mean_168h', 'rolling_std_24h']].head()


Required features present:
  consumption_kwh_lag_1h: ✓
  consumption_kwh_lag_24h: ✓
  rolling_mean_168h: ✗ MISSING
  rolling_std_24h: ✗ MISSING
  hdd: ✓

Reconstructed 'region' column from one-hot encoding

Regions: ['Annapolis Valley', 'Cape Breton', 'Halifax', 'Pictou County', 'South Shore']

Sample data:


KeyError: "['rolling_mean_168h', 'rolling_std_24h'] not in index"

## Prophet Forecasting - All Regions with Regressors

Features used as regressors:
- consumption_kwh (target)
- Lag features: lag_1h, lag_24h
- Rolling statistics: rolling_mean_168h, rolling_std_24h
- hdd (Heating Degree Days)
- is_holiday

We'll forecast each region separately with 80/20 train/test split.

In [ ]:
def forecast_region_prophet(region_name, df_full, max_years=None):
    """Run Prophet forecast for one region"""
    print(f"{'='*70}")
    print(f"Prophet Forecasting: {region_name}")
    print(f"{'='*70}")
    # Filter region
    region_df = df_full[df_full['region'] == region_name].copy()
    region_df = region_df.sort_values('timestamp').reset_index(drop=True)

    # Optionally limit to recent years to keep training time reasonable
    if max_years is not None and len(region_df) > 0:
        hours = int(24 * 365 * max_years)
        if len(region_df) > hours:
            print(f"Truncating to last {max_years} years ({hours} rows) for speed.")
            region_df = region_df.tail(hours).reset_index(drop=True)

    prophet_df = region_df[['timestamp', 'consumption_kwh', 
                            'temperature_c', 'humidity_pct', 
                            'hdd', 'consumption_kwh_lag_1h', 'consumption_kwh_lag_24h',
                            'rolling_mean_168h', 'rolling_std_24h',
                            'is_holiday']].copy()
    prophet_df.columns = ['ds', 'y', 'temperature', 'humidity', 'hdd',
                         'lag_1h', 'lag_24h', 'rolling_mean_168h', 'rolling_std_24h',
                         'is_holiday']

    # Train/test split (80/20)
    split_idx = int(len(prophet_df) * 0.8)
    train_df = prophet_df[:split_idx].copy()
    test_df = prophet_df[split_idx:].copy()

    print(f"Train: {len(train_df)} rows ({train_df['ds'].min()} to {train_df['ds'].max()})")
    print(f"Test:  {len(test_df)} rows ({test_df['ds'].min()} to {test_df['ds'].max()})")

    model = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=True, seasonality_mode='multiplicative')
    model.add_regressor('temperature')
    model.add_regressor('humidity')
    model.add_regressor('hdd')
    model.add_regressor('lag_1h')
    model.add_regressor('lag_24h')
    model.add_regressor('rolling_mean_168h')
    model.add_regressor('rolling_std_24h')
    model.add_regressor('is_holiday')

    print('Training Prophet model...')
    model.fit(train_df)

    forecast = model.predict(test_df)

    calculate_metrics(test_df['y'].values, forecast['yhat'].values, 'Prophet', region_name)

    export_df = forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
    export_df['model'] = 'Prophet'
    export_df['region'] = region_name

    return export_df, model, forecast, test_df

In [ ]:
# Forecast all regions
regions = sorted(df['region'].unique())
print(f"Forecasting {len(regions)} regions: {regions}")

prophet_forecasts = []

for region in regions:
    # limit to last 3 years to keep runtime reasonable (adjustable)
    try:
        forecast_df, model, forecast, test_df = forecast_region_prophet(region, df, max_years=3)
        prophet_forecasts.append(forecast_df)
    except Exception as e:
        print(f"Error running Prophet for {region}: {e}")

# Combine
if prophet_forecasts:
    prophet_combined = pd.concat(prophet_forecasts, ignore_index=True)
    print(f"✓ Prophet forecasting complete: {len(prophet_combined)} predictions")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    prophet_combined.to_csv(OUTPUT_DIR / 'prophet_forecasts.csv', index=False)
    # Save Prophet metrics only
    try:
        metrics_df = pd.DataFrame(model_results)
        metrics_df[metrics_df['Model']=='Prophet'].to_csv(OUTPUT_DIR / 'prophet_metrics_summary.csv', index=False)
        # also save full model results for record
        metrics_df.to_csv(OUTPUT_DIR / 'model_results.csv', index=False)
        print(f"Saved Prophet forecasts and metrics to {OUTPUT_DIR}")
    except Exception as e:
        print(f"Could not save metrics: {e}")
else:
    print('No Prophet forecasts generated.')

## LSTM Forecasting - Multi-Region with Multivariate Features

Features for LSTM:
- consumption_kwh (target)
- Lag features: lag_1h, lag_24h
- Rolling statistics: rolling_mean_168h, rolling_std_24h
- Weather: temperature_c, humidity_pct
- Time features: hour, day_of_week, is_weekend
- Grid features: grid_load_pct, renewable_pct

We'll use PyTorch LSTM with multivariate sequences.

In [ ]:
class LSTMForecaster(nn.Module):
    def __init__(self, input_size, hidden_size=100, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                           dropout=dropout, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_output = lstm_out[:, -1, :]
        prediction = self.fc(last_output)
        return prediction

In [ ]:
def forecast_region_lstm(region_name, df_full, sequence_length=24, epochs=20, batch_size=32, subsample_years=2, device='cuda'):
    region_df = df_full[df_full['region'] == region_name].copy()
    region_df = region_df.sort_values('timestamp').reset_index(drop=True)

    # Optional: subsample to last N years for speed
    if subsample_years is not None:
        region_df = region_df.tail(365 * 24 * int(subsample_years))
        print(f"Using last {subsample_years} years ({len(region_df)} rows) for training speed")

    if 'hour' not in region_df.columns:
        region_df['hour'] = region_df['timestamp'].dt.hour
    if 'day_of_week' not in region_df.columns:
        region_df['day_of_week'] = region_df['timestamp'].dt.dayofweek
    if 'is_weekend' not in region_df.columns:
        region_df['is_weekend'] = region_df['day_of_week'].isin([5,6]).astype(int)

    feature_cols = ['consumption_kwh', 'consumption_kwh_lag_1h', 'consumption_kwh_lag_24h',
                   'rolling_mean_168h', 'rolling_std_24h',
                   'temperature_c', 'humidity_pct',
                   'hour', 'day_of_week', 'is_weekend',
                   'grid_load_pct', 'renewable_pct']

    region_df = region_df.dropna(subset=feature_cols).reset_index(drop=True)
    if len(region_df) < sequence_length + 10:
        print(f"{region_name}: insufficient data after dropna; skipping")
        return None, None, None, None, None

    data = region_df[feature_cols].values
    scaler = MinMaxScaler()
    data_scaled = scaler.fit_transform(data)

    X, y = [], []
    for i in range(len(data_scaled) - sequence_length):
        X.append(data_scaled[i:i+sequence_length])
        y.append(data_scaled[i+sequence_length, 0])
    X, y = np.array(X), np.array(y).reshape(-1,1)

    split_idx = int(len(X) * 0.8)
    X_train, X_test = X[:split_idx], X[split_idx:]
    y_train, y_test = y[:split_idx], y[split_idx:]
 
    print(f"Device:  {device}")
    X_train_t = torch.FloatTensor(X_train)
    y_train_t = torch.FloatTensor(y_train)
    X_test_t = torch.FloatTensor(X_test)
    y_test_t = torch.FloatTensor(y_test)

    model = LSTMForecaster(input_size=len(feature_cols), hidden_size=50, num_layers=1)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    # Batch training with DataLoader
    from torch.utils.data import TensorDataset, DataLoader
    train_dataset = TensorDataset(X_train_t, y_train_t)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    model.train()
    for epoch in range(epochs):
        epoch_loss = 0
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        avg_loss = epoch_loss / len(train_loader)
        if (epoch + 1) % 5 == 0:
            print(f"  Epoch [{epoch+1}/{epochs}], Avg Loss: {avg_loss:.4f}")

    # Evaluate in batches
    test_dataset = TensorDataset(X_test_t, y_test_t)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    predictions = []
    model.eval()
    with torch.no_grad():
        for batch_X, _ in test_loader:
            batch_pred = model(batch_X)
            predictions.append(batch_pred)

    if predictions:
        y_pred_t = torch.cat(predictions, dim=0)
    else:
        y_pred_t = model(X_test_t)

    y_test_actual = scaler.inverse_transform(np.column_stack([y_test, np.zeros((len(y_test), len(feature_cols)-1))]))[:,0]
    y_pred_actual = scaler.inverse_transform(np.column_stack([y_pred_t.numpy(), np.zeros((len(y_pred_t), len(feature_cols)-1))]))[:,0]

    # Correct calculate_metrics call with model and region
    mae, rmse, mape = calculate_metrics(y_test_actual, y_pred_actual, 'LSTM', region_name)
    print(f"{region_name}: MAE={mae:.2f}, RMSE={rmse:.2f}, MAPE={mape:.2f}%")

    # Build LSTM export DataFrame with timestamps aligned to test targets
    try:
        test_start_idx = split_idx + sequence_length
        n_test = len(y_test_actual)
        test_timestamps = region_df.iloc[test_start_idx:test_start_idx + n_test]['timestamp'].values
        std_pred = np.std(y_pred_actual) if len(y_pred_actual)>0 else 0.0
        lstm_export = pd.DataFrame({
            'ds': test_timestamps,
            'yhat': y_pred_actual,
            'yhat_lower': y_pred_actual - (2 * std_pred),
            'yhat_upper': y_pred_actual + (2 * std_pred),
            'model': 'LSTM',
            'region': region_name
        })
    except Exception as e:
        print(f"Could not build LSTM export for {region_name}: {e}")
        lstm_export = None

    return y_test_actual, y_pred_actual, model, scaler, lstm_export

In [ ]:
# Train LSTM for all regions (quick run: epochs=5)
lstm_results = {}
lstm_forecasts = []

for region in regions:
    try:
        y_test, y_pred, model, scaler, lstm_export = forecast_region_lstm(region, df, epochs=5, device=DEVICE)
        if y_test is not None:
            lstm_results[region] = {'y_test': y_test, 'y_pred': y_pred}
            if lstm_export is not None:
                lstm_forecasts.append(lstm_export)
    except Exception as e:
        print(f"Error training LSTM for {region}: {e}")

if lstm_forecasts:
    lstm_combined = pd.concat(lstm_forecasts, ignore_index=True)
    print(f"\n✓ LSTM forecasting complete: {len(lstm_combined)} predictions")
    try:
        lstm_combined.to_csv(OUTPUT_DIR / 'lstm_forecasts.csv', index=False)
        print(f"Saved LSTM forecasts: {OUTPUT_DIR / 'lstm_forecasts.csv'}")
    except Exception as e:
        print(f"Could not save lstm_combined: {e}")
else:
    print('No LSTM forecasts generated.')

# Save LSTM metrics to OUTPUT_DIR (if calculate_metrics appended results)
try:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    metrics_df = pd.DataFrame(model_results)
    if 'LSTM' in metrics_df['Model'].unique():
        metrics_df[metrics_df['Model']=='LSTM'].to_csv(OUTPUT_DIR / 'lstm_metrics_summary.csv', index=False)
        print(f"Saved LSTM metrics to {OUTPUT_DIR / 'lstm_metrics_summary.csv'}")
    else:
        print('No LSTM metrics found in model_results to save.')
except Exception as e:
    print(f"Could not save LSTM metrics: {e}")

## Export Forecast Results for Tableau Dashboard

Combining Prophet and LSTM forecasts for all regions.

In [ ]:
# Export Prophet forecasts
try:
    prophet_combined.to_csv(OUTPUT_DIR / 'prophet_forecasts_all_regions.csv', index=False)
    print(f"Saved Prophet forecasts: {len(prophet_combined)} rows")
except Exception as e:
    print(f"Could not save prophet_combined: {e}")

# Note: LSTM full time-aligned export requires reconstructed timestamps from sequences.
# If `lstm_results` contains per-region test arrays, we can save a simple summary file.
try:
    # Combine metrics from both models
    results_df = pd.DataFrame(model_results)
    results_df = results_df.sort_values(['Region', 'Model'])
    print("\n" + "="*70)
    print("FINAL MODEL COMPARISON - ALL REGIONS")
    print("="*70)
    print(results_df.to_string(index=False))
    # Save metrics
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    results_df.to_csv(OUTPUT_DIR / 'forecast_metrics_all_regions.csv', index=False)
    print(f"Saved metrics: {OUTPUT_DIR / 'forecast_metrics_all_regions.csv'}")
    # Create visualization (may fail in headless env)
    try:
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        for idx, metric in enumerate(['MAE', 'RMSE', 'MAPE']):
            pivot = results_df.pivot(index='Region', columns='Model', values=metric)
            pivot.plot(kind='bar', ax=axes[idx], rot=45)
            axes[idx].set_title(f'{metric} by Region and Model')
            axes[idx].set_ylabel(metric)
            axes[idx].legend(title='Model')
        plt.tight_layout()
        save_and_show_plot(fig, 'forecast_comparison_all_regions.png')
    except Exception as e:
        print(f"Could not create/save visualization: {e}")

    # Best model per region
    print("\n" + "="*70)
    print("BEST MODEL BY REGION (Lowest MAE)")
    print("="*70)
    best_models = results_df.loc[results_df.groupby('Region')['MAE'].idxmin()]
    print(best_models[['Region', 'Model', 'MAE', 'RMSE', 'MAPE']].to_string(index=False))
except Exception as e:
    print(f"Could not build final comparison: {e}")